In [ ]:
import os

import pandas as pd

from pynxtools_microstructure.examples.oasisb.batch_process import process_project
from pynxtools_microstructure.examples.oasisb.oasisb_utils import get_project_id

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)
with open("alias_prefix_secret.txt") as fp:
    alias_prefix_secret: str = f"{fp.readline().strip().replace('/', os.sep)}"
print(alias_prefix_secret)

os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
# os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")
project_range: tuple[int, int] = (1, 1)

count: int = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                process_project(
                    project_id,
                    # f"{src_directory}{os.sep}{project_id}.ods",
                    f"{src_directory}{os.sep}aaa_legacy_data.bib",
                    f"{trg_directory.replace('/mtex', '/decompressed')}{os.sep}{project_id}.decompressed.log",
                    trg_directory,
                    f"{trg_directory.replace('/mtex', '/pynxtools')}",
                    alias_prefix_secret,
                    openalex_file=f"{os.getcwd()}{os.sep}openalex/D{project_id}.json",
                    logger_file_path_suffix="test",
                )
                count += 1
print(f"Batch queue completed {count}")

***